# Raster operations: Local, Focal, Global, and Zonal 

This session organizes operations performed on raster data into four classes:

- **Local**: each output cell depends on the same cell of the input. Unit conversion, boolean conditions, reclassification. Otherwise known as raster algebra
- **Focal**: each output cell depends on a neighborhood of the input. A moving-window mean, a focal range, and slope, which is the rate of change across a cell's neighbors
- **Global**: the whole raster reduces to a number, such as its minimum, mean, or a percentile
- **Zonal**: the raster is summarized within each polygon of a vector layer

By the end of this session you will be able to:

- Classify a raster operation as local, focal, global, or zonal, and say why the distinction matters for nodata handling
- Compute global statistics and use one in a local operation
- Build a 3-by-3 moving window with numpy and compute a focal mean and a focal range with nodata handled correctly
- Apply raster algebra: convert units, combine two rasters with a boolean condition, and reclassify a continuous raster into named classes
- Compute a slope raster with numpy's gradient and explain what slope in degrees means at the pixel level
- Run zonal statistics with rasterstats and recognize the edge case of a zone with zero raster pixels

## Today's structure

**Part 1, lecture (about 15 minutes).** The four kinds of raster operation and how each one functions.

**Part 2, in teams of two or three (the rest of the session).** Five problems (one per operation), with slope as a second focal problem, building on each other. For every problem:

1. **Read before running.** Open the code cell and read it line by line as a team. Agree on what each line does.
2. **Annotate.** Lines marked `# ANNOTATE` in the problem code carry no explanation. Replace each marker with a comment in your own words stating what the line does and why it is there.
3. **Predict.** Answer the "Predict" question and write the team's prediction in the markdown cell provided.
4. **Run and check.** Run the cell. Decide as a team whether the output is plausible, using the checks from the lecture: value ranges before and after, counts that should agree with each other, and a plot. If something looks wrong, determine why before moving on.
5. **Modify, then write.** Make the change the problem's "Modify" task asks for, re-run, and explain what changed and why. Then complete the exercise in a new code cell below it.

Lab 5 requires these operations on a different DEM, so completing these tasks will help you finish your lab. A readiness check at the end maps each problem to a lab task.

In [ ]:
# Run this cell first (takes ~1 minute). Colab already includes geopandas.
%pip install -q libpysal esda splot mapclassify contextily
%pip install -q rasterio rioxarray rasterstats

## Part 1 (lecture): Four kinds of raster operations

Every operation on a raster can be sorted by one question: **what part of the input does a single output cell depend on?** 

Tomlin (1990) used that question to divide map algebra into four classes.

| Class | One output cell depends on | Examples | numpy or library tool |
|---|---|---|---|
| **Local** | the same cell of one or more input rasters | unit conversion, thresholds, reclassification, combining two rasters | array arithmetic, comparisons, `np.where`, `np.digitize` |
| **Focal** | a neighborhood of cells around the same position | moving-window mean, focal range, slope, aspect, hillshade | `sliding_window_view`, `np.gradient` |
| **Global** | every cell of the raster | minimum, maximum, mean, percentile, count | `np.nanmin`, `np.nanmean`, `np.nanpercentile` |
| **Zonal** | every cell inside an irregular zone defined by a polygon | mean elevation per county, pixel count per municipality | `rasterstats.zonal_stats` |

Last session, we calcualted a mean, a minimum, a count of nodata cells. Which type of operation were those?



### Local operations

A local operation is applied cell by cell. 

The output at row *r*, column *c* depends only on the input at row *r*, column *c*, so the operation can be written as if the raster were a single number. 

Raster algebra is the classical case: `dem * 2`, `dem - 4`, `dem ** 2`, `np.log(dem)`. 

Any arithmetic expression works, so does any function that accepts an array, which is how we can apply a transformation such as NOAA's cyanobacteria-index conversion (`10 ** (0.011714 * DN - 4.1870866)`) to a whole scene in one line.

Three further forms of local operation appear constantly:

- **A condition.** `dem > 380` produces a boolean raster, `True` where the condition holds. Its sum is a count of cells, and a count times the cell area is an area on the ground.
- **A reclassification.** A set of rules or conditions turns a continuous raster into a few classes: elevation into valley, slope, and ridge; a water-quality index into absent, present, and severe. `np.digitize` does this in numpy.
- **Combining two rasters.** `(dem > 380) & (slope > 10)` asks a question of two rasters simultaneously. Multiplying a raster by a binary raster of ones and zeros keeps the cells under the ones and zeroes the rest, which is masking by arithmetic.

Two rules follow from the definition:

1. First, a local operation on two rasters requires that they align exactly: the same shape, extent, resolution, CRS, and origin, so that row *r*, column *c* means the same place in both. If the two came from different files, confirm that before combining them; numpy will happily multiply two arrays of the same shape that cover different ground. 
2. Second, a local operation can always be checked with a global one. For exmaple, compute the value range before and after and confirm the change is what the arithmetic predicts. If `dem * 2` doesn't double the range, the operation did not do what was intended.

### Focal operations

A focal operation computes each output cell from the cells around it. 

The neighborhood goes by several names (window, kernel, filter, moving window) and is most often the 3-by-3 block of a cell and its eight neighbors, though any size or shape is possible.

An aggregation function reduces the window to one value, that value becomes the output at the center cell, and the window slides one cell over and repeats.

A 3-by-3 mean on one position of a grid:

```
input window        output
 300  302  305
 298  301  304   ->  301.6   (the mean of all nine, written to the center)
 297  299  309
```

The same idea on a real neighborhood, with the window shown at one position and the aggregation written to the center:

![A 3-by-3 moving window over a raster, with the output written to the center cell](https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/notebooks/images/focal_example.png)

*Figure from Lovelace, Nowosad, and Muenchow, [Geocomputation with R](https://r.geocompx.org/spatial-operations.html#map-algebra), section 4.3.4.*

Two consequences of the definition matter in practice:

- **Edge effects:** A cell on the raster's border has no neighbors on one side. Something must stand in for them: 
    - the window can shrink
    - the raster can be padded with nodata that the aggregation ignores
    - the edge rows can be dropped from the output. 

The method taught this session pads with `nan` and reduces with functions that ignore `nan`, which makes the edge cells and the interior nodata cells behave the same way.

- **Nodata "spreads".** If the aggregation function does not ignore nodata, one nodata cell "poisons" the eight outputs around it, and a nodata boundary becomes a two-cell-wide band of nodata in the result. Focal operations are where nodata handling matters most.

Slope, aspect, and hillshade are all **focal**: each is computed from the elevations of a cell's neighbors. So is any smoothing or roughness measure. 

***A focal operation is not the same as changing resolution:*** aggregating a raster to larger cells produces fewer cells, while a focal operation produces an output the same size as the input, with every cell replaced by a summary of its surroundings.

One way to see what a focal operation did is to subtract the input from the output. The difference raster is itself a **local** operation on two aligned rasters, and it shows where the window changed values most, which for a smoothing filter is wherever the terrain is roughest.

### Global operations

A global operation reduces the whole raster to a single number: 
- minimum
- maximum
- mean
- standard deviation
- percentile
- count of cells meeting a condition. 

On their own they are the least interesting of the four classes. They matter because they feed the others. A threshold at the 90th percentile, a standardization by the global mean and standard deviation, or a share of cells above the mean each begins with a global statistic and ends with a local operation that uses it.

The nodata rule is the one from the previous session: a global statistic over an array that contains nodata is wrong, either absurd (a mean dragged toward `-9999`) or simply `nan`. 

Mask first, or use the `np.nan*` functions, which treat `nan` as absent.

### Zonal operations

A zonal operation is a focal operation with an irregular neighborhood. Instead of a fixed window around each cell, the neighborhoods are zones, defined by a second raster of zone codes or, more often, by polygons from a vector layer. 

Each zone receives one summary of every raster cell inside it: a mean, a count, a maximum. The output is a table with one row per zone, or a vector layer with new columns, rather than a raster.



### The same nodata, four different failures

The categories are worth learning because each has its own nodata rule, and the same `-9999` cells cause a different failure in each. 

A local operation on a `nan` cell yields `nan`, which is correct and harmless. 

A focal operation considers every cell's neighbors, so one `nan` can spread into eight neighboring outputs unless the window reduction ignores it. 

A global statistic over an array that contains `nan` is `nan`, or wildly wrong if the sentinel was never converted.

A zonal statistic silently includes the sentinel in a zone's mean unless the nodata value is declared to `rasterstats`. 

Therefore the first step in every problem below is the same: convert nodata to `nan` before doing anything else.

## Review: 

**Classify each of the following as local, focal, global, or zonal:**

1. converting elevation from meters to feet
2. averaging each cell with its eight neighbors
3. finding the county's highest elevation
3. computing slope
4. computing mean elevation per municipality.


## Part 2: Problems

## Problem 1: Global operations (team work)

This problem also loads the DEM and defines the variables all other problems use: `dem_f`, `valid`, `cellsize`, `transform`, and `nodata`. Run it first.

**Predict:** the county's elevation runs from 261 to 428 m. Will more than half, less than half, or about half of the cells lie above the mean elevation? Say why.


### Workthrough and annotate

In [ ]:
DATA_BASE_P2 = "https://raw.githubusercontent.com/pjbitterman/KSU_x9076/main/data/part2"

# Local path fallback (uncomment if running outside Colab without internet access):
# DATA_BASE_P2 = "../data/part2"

import rasterio
import numpy as np

dem_path = f"{DATA_BASE_P2}/portage_dem.tif"

with rasterio.open(dem_path) as src:
    dem = src.read(1)
    nodata = src.nodata
    transform = src.transform
    cellsize = src.res[0]

dem_f = dem.astype(float)
# ANNOTATE: what does this line do, and why must it happen before anything else?
dem_f[dem == nodata] = np.nan
# ANNOTATE: what does valid hold, and what is it used for later?
valid = ~np.isnan(dem_f)

# ANNOTATE: why these four functions and not min(), max(), mean(), and percentile()?
dem_min, dem_max, dem_mean = np.nanmin(dem_f), np.nanmax(dem_f), np.nanmean(dem_f)
dem_p90 = np.nanpercentile(dem_f, 90)
print(f"global: min {dem_min:.1f}  max {dem_max:.1f}  mean {dem_mean:.1f}  90th percentile {dem_p90:.1f} m")

# ANNOTATE: which class of operation is this line, and which class produced the number it compares against?
above_mean = dem_f > dem_mean
print(f"cells above the county mean: {100 * above_mean.sum() / valid.sum():.1f}% of valid cells")


### Modify 

Replace the 90th percentile with the median (`np.nanmedian`) and the standard deviation (`np.nanstd`), then change the `above_mean` threshold from the mean to the median. What share of cells lies above the median, why is that number not a surprise, and what does the difference between the mean and the median say about the shape of the elevation distribution?

-----

## Problem 2: Focal operations (team work)

**Predict:** a 3-by-3 mean replaces each cell with the average of itself and its neighbors. On terrain where adjacent cells differ by a few meters, how much does that change a typical cell: a few centimeters, a few meters, or tens of meters? Where would the largest changes occur?


### Background: a moving window

A focal operation evaluates a window centered on each cell. The 3-by-3 window is the smallest: the cell and its eight neighbors. numpy builds all of the windows at once with `sliding_window_view`, which returns a four-dimensional view with shape `(rows, cols, 3, 3)` without copying the array. Padding the array with one ring of `nan` first gives the edge cells a full window, and reducing each window with `np.nanmean`, `np.nanmax`, or `np.nanmin` ignores the padding and any nodata inside the window. The one remaining step is to restore `nan` on the cells that were nodata to begin with, since their windows may contain valid neighbors.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

# ANNOTATE: why pad, why by one cell, and why with nan rather than zero?
padded = np.pad(dem_f, 1, constant_values=np.nan)
# ANNOTATE: what is the shape of windows, and does this line copy the array?
windows = sliding_window_view(padded, (3, 3))
print("window array shape:", windows.shape)

# ANNOTATE: what do axis=(2, 3) select, and what would np.mean do here instead of np.nanmean?
focal_mean = np.nanmean(windows, axis=(2, 3))
# ANNOTATE: the cells this line changes already had a value from the line above. Why replace it?
focal_mean = np.where(valid, focal_mean, np.nan)

# ANNOTATE: what does this quantity measure on a DEM, in what units?
focal_range = np.nanmax(windows, axis=(2, 3)) - np.nanmin(windows, axis=(2, 3))
focal_range = np.where(valid, focal_range, np.nan)

diff = np.abs(focal_mean - dem_f)
print(f"focal mean vs original: mean change {np.nanmean(diff):.2f} m, largest change {np.nanmax(diff):.1f} m, "
      f"{100 * np.nansum(diff > 5) / valid.sum():.1f}% of cells change by more than 5 m")
print(f"focal range: mean {np.nanmean(focal_range):.1f} m, max {np.nanmax(focal_range):.1f} m, "
      f"95th percentile {np.nanpercentile(focal_range, 95):.1f} m")
print(f"cells with a focal range of 10 m or more: {100 * np.nansum(focal_range >= 10) / valid.sum():.1f}%")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(np.ma.masked_invalid(focal_range), cmap="magma")
fig.colorbar(im, ax=ax, label="3x3 focal range (m)")
ax.set_title("Portage County roughness: elevation range within each 3x3 window")
plt.show()


### Modify and extend

1. change the reduction for `focal_mean` from `np.nanmean` to `np.nanmedian` and re-run. Does the median move cells by more or by less than the mean did, and why would a median filter be preferred on a raster with occasional bad cells?

2. Build a 5-by-5 focal mean: pad by 2 and request a `(5, 5)` window. Report the mean and largest change from the original. Does a wider window change cells by more or by less than the 3-by-3 did, and what happens to the stream valleys on a plot of the 5-by-5 focal range?

-----

## Problem 3: Local operations, raster algebra (team work)

Three code cells follow, one per form of raster algebra: scalar arithmetic, boolean conditions, reclassification.


### Background: the three forms

Raster algebra is the local operation: arithmetic on every cell at once, with each output cell depending only on the same input cell. Three forms:

- **Scalar arithmetic**: `dem_f * 3.28084` converts every elevation to feet; `dem_f - np.nanmin(dem_f)` gives each cell's height above the lowest cell. One expression, no loop.
- **Boolean conditions**: `focal_range >= 10` produces a raster of `True` and `False`, one per cell. Two boolean rasters combine with `&` (both) and `|` (either), which is how a question such as "high *and* rough" is asked of two rasters at once. The count of `True` cells, multiplied by the cell area, is an area on the ground.
- **Reclassification**: `np.digitize()` assigns every cell to a class by where its value falls among a set of edges, turning a continuous raster into a few named categories.

Nodata behaves consistently through all three, because the nodata cells are `nan`: arithmetic yields `nan` and comparison yields `False`. Two consequences follow. Plain `.mean()` on such an array returns `nan`, so the `np.nan*` functions are required, and `np.digitize()` places `nan` in the last class unless it is handled explicitly.

In [ ]:
# ANNOTATE: how many cells does this line touch, and what happens on the nodata cells?
dem_ft = dem_f * 3.28084
# ANNOTATE: which class of operation is np.nanmin here, and which class is the subtraction?
relief = dem_f - np.nanmin(dem_f)

print(f"elevation in feet: {np.nanmin(dem_ft):.0f}-{np.nanmax(dem_ft):.0f} ft")
print(f"relief above the lowest cell: up to {np.nanmax(relief):.1f} m, mean {np.nanmean(relief):.1f} m")
# ANNOTATE: predict what this line prints before running it, and say why
print("plain .mean() on an array that holds nan:", dem_ft.mean())


In [ ]:
# ANNOTATE: why is cellsize squared, and where did cellsize come from?
cell_area_km2 = cellsize ** 2 / 1e6

high = dem_f > 380
rough = focal_range >= 10
# ANNOTATE: what does & do between two arrays, and why would the keyword 'and' fail here?
high_and_rough = high & rough

for name, mask in [("high (> 380 m)", high), ("rough (range >= 10 m)", rough), ("high and rough", high_and_rough)]:
    # ANNOTATE: a comparison with nan is False. Why does that make a separate nodata mask unnecessary here?
    n = mask.sum()
    print(f"{name:<24} {n:>9,} cells  {100 * n / valid.sum():5.1f}% of county  {n * cell_area_km2:6.1f} km2")


In [ ]:
bins = [5, 10, 20]
labels = ["smooth (<5)", "gentle (5-10)", "rough (10-20)", "very rough (>=20)"]

# ANNOTATE: what integer does np.digitize assign to a value of 12? To a value of 3? To nan?
rough_class = np.digitize(focal_range, bins)
# ANNOTATE: what would the class counts below report without this line?
rough_class = np.where(np.isnan(focal_range), -1, rough_class)

for k, label in enumerate(labels):
    n = (rough_class == k).sum()
    print(f"class {k}  {label:<18} {100 * n / valid.sum():5.1f}%")

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(np.ma.masked_equal(rough_class, -1), cmap="YlOrBr", vmin=0, vmax=3)
cbar = fig.colorbar(im, ax=ax, ticks=range(4))
cbar.ax.set_yticklabels(labels)
ax.set_title("Portage County roughness, four classes")
plt.show()


### Modify and extend

1. in the second cell, change `&` to `|` and re-run; explain the new count from the two that feed it. In the third cell, delete the `np.where` line that follows `np.digitize` and re-run; explain what happened to the top class and why.
2. build a boolean raster for valley floor: elevation below `300` m and focal range below `2` m. Report its cell count, its share of the county, and its area in km². Then reclassify elevation itself into three bands with edges at `300` and `380` m and report the share in each.

-----

## Problem 4: Slope, a second focal operation (team work)

**Predict:** Portage County is glaciated till plain in northeast Ohio, resulting in rolling (not mountainous) topography, with 167 m of total relief across the DEM's 261 to 428 m range. Estimate the typical pixel-to-pixel slope: closer to 1 to 2 degrees, 5 to 10 degrees, or more than 20?


### Background: slope from a gradient

Slope measures how quickly elevation changes between a cell and its neighbors. It is a focal operation, and `numpy.gradient()` computes the rate of change along each axis of an array from adjacent cells. Because `portage_dem.tif` is in UTM meters, both axes carry real distance units and the gradient converts directly to a slope in degrees. Slope improves on the focal range as a measure of steepness in two ways: it is normalized by the cell size, so it does not depend on the window, and it has physical units that transfer between rasters of different resolution.

_Team prediction:_

In [ ]:
# ANNOTATE: what are dzdy and dzdx, what is cellsize doing in this call, and why must nodata already be nan?
dzdy, dzdx = np.gradient(dem_f, cellsize)
# ANNOTATE: in words, what does this line do to turn two gradients into one angle?
slope_deg = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))

print(f"slope range: {np.nanmin(slope_deg):.1f}-{np.nanmax(slope_deg):.1f} degrees")
print(f"mean slope: {np.nanmean(slope_deg):.1f} degrees")


## A second worked example: windowed slope, flat versus dissected

The full-grid `slope_deg` summarizes the county in one number, a `5.0`-degree mean. The windowed reads from the previous session showed that a DEM this size contains internal variation, and the same `Window` mechanism can pull two samples for direct comparison: do "flat" and "dissected" describe different places, or the same terrain under different names?

**Predict:** the two 300-by-300 windows below share the column offset of the previous session's center window (`col_off=600`), one further south (`row_off=1200`) and one further north (`row_off=150`). Given a `5.0`-degree county-wide mean and 167 m of relief, predict whether a subregion's mean slope varies much from window to window. After running, pick a third window location of your own and add it to the comparison.

In [ ]:
from rasterio.windows import Window

def window_slope(row_off, col_off, size=300):
    window = Window(col_off=col_off, row_off=row_off, width=size, height=size)
    with rasterio.open(dem_path) as src:
        sub = src.read(1, window=window).astype(float)
        sub[sub == src.nodata] = np.nan
        res = src.res[0]
    dy, dx = np.gradient(sub, res)
    return np.degrees(np.arctan(np.sqrt(dx**2 + dy**2))), sub

flat_slope, flat_elev = window_slope(row_off=1200, col_off=600)
dissected_slope, dissected_elev = window_slope(row_off=150, col_off=600)

print(f"flat window       -- mean slope {np.nanmean(flat_slope):.1f} deg, max {np.nanmax(flat_slope):.1f} deg, "
      f"elevation range {np.nanmin(flat_elev):.1f}-{np.nanmax(flat_elev):.1f} m")
print(f"dissected window  -- mean slope {np.nanmean(dissected_slope):.1f} deg, max {np.nanmax(dissected_slope):.1f} deg, "
      f"elevation range {np.nanmin(dissected_elev):.1f}-{np.nanmax(dissected_elev):.1f} m")

In [ ]:
slope_bins = [2, 5, 10]
slope_labels = ["flat (<2)", "gentle (2-5)", "moderate (5-10)", "steep (>=10)"]
# ANNOTATE: this line does in one step what Problem 3 did in two. Which two?
slope_class = np.where(np.isnan(slope_deg), -1, np.digitize(slope_deg, slope_bins))

for k, label in enumerate(slope_labels):
    n = (slope_class == k).sum()
    print(f"class {k}  {label:<16} {100 * n / valid.sum():5.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
im0 = axes[0].imshow(np.ma.masked_invalid(slope_deg), cmap="viridis")
fig.colorbar(im0, ax=axes[0], label="slope (degrees)")
axes[0].set_title("Slope")
im1 = axes[1].imshow(np.ma.masked_invalid(focal_range), cmap="magma")
fig.colorbar(im1, ax=axes[1], label="3x3 focal range (m)")
axes[1].set_title("Focal range, for comparison")
plt.show()


### Modify and extend

1. in the first cell, remove `cellsize` from the `np.gradient` call and re-run. What happens to the slope values, and what unit is the gradient in without it? Restore it. Then, in the reclassification cell, change the edges to `[5, 15]` with three labels, which is the scheme Lab 5 uses, and report the new amounts of each class.
2. Plot `slope_deg` with `imshow()` and a colorbar on its own, using `np.ma.masked_invalid(slope_deg)`. Then convert slope to percent grade with `np.tan(np.radians(slope_deg)) * 100` and report the maximum. Is a 38.8-degree slope closer to a 40% grade or an 80% grade?


-----

## Problem 5: Zonal statistics (team work)


### Predict

`portage_munis.geojson` holds 23 municipalities that touch Portage County's boundary. Some lie entirely inside it; others (Tallmadge, Hudson, Aurora, Solon, among others) lie mostly in a neighboring county with a small part crossing into Portage. Does every one of the 23 receive a mean elevation from `zonal_stats()`, or could one return empty?


### Background: the raster-to-vector bridge

Every operation so far stayed within the grid. Zonal statistics, the fourth category, crosses to vector data: given a polygon layer (the zones) and a raster, `rasterstats.zonal_stats()` reports summary statistics (mean, minimum, maximum, count) of every raster cell inside each polygon. It is the raster counterpart of week 9's spatial join: instead of attaching a point's containing county, it attaches a summary of every pixel inside a polygon.

In [ ]:
import geopandas as gpd
from rasterstats import zonal_stats

munis = gpd.read_file(f"{DATA_BASE_P2}/portage_munis.geojson")
print(munis.crs, munis.shape)

# ANNOTATE: why this CRS, and what would zonal_stats do with the polygons if this line were skipped?
munis_p = munis.to_crs("EPSG:26917")

# ANNOTATE: state what each of the five arguments supplies, and why dem (not dem_f) is passed here
stats = zonal_stats(munis_p, dem, affine=transform, nodata=nodata, stats=["mean", "min", "max", "count"])
# ANNOTATE: what is stats (its type and length), and what does this line turn it into?
munis_p[["mean_elev_m", "min_elev_m", "max_elev_m", "pixel_count"]] = [
    [s["mean"], s["min"], s["max"], s["count"]] for s in stats
]

munis_p[["name", "mean_elev_m", "min_elev_m", "max_elev_m", "pixel_count"]].sort_values(
    "mean_elev_m", ascending=False
)


### Modify and extend

1. add `"median"` and `"std"` to the `stats` list in the first cell and store them as two more columns. Which municipality has the largest gap between its mean and its median elevation, and what does a large gap say about the terrain inside it? Then re-run the first cell with the `to_crs` line commented out and explain the result.
2. `munis_p[munis_p["pixel_count"] == 0]` isolates any municipality that received no cells. Identify it, explain from the data why it received none, then drop it with `munis_p[munis_p["pixel_count"] > 0]` and confirm how many rows remain.

----

## A second worked example: zonal statistics on slope

`zonal_stats()` is indifferent to what the raster's values represent. The same municipalities, the same affine transform, and a different raster produce a different summary. Here it runs on `slope_deg` instead of elevation, asking not how high each municipality is but how steep it is on average.

### Predict

one municipality had the highest mean elevation in the table above. Does the municipality with the highest mean elevation also have the highest mean slope, or are the two independent properties of a landscape?

In [ ]:
# ANNOTATE: slope_deg already holds nan for nodata. Why does zonal_stats need -9999 instead, and why float32?
slope_filled = np.where(np.isnan(slope_deg), -9999, slope_deg).astype("float32")

slope_stats = zonal_stats(munis_p, slope_filled, affine=transform, nodata=-9999, stats=["mean", "max"])
munis_p[["mean_slope_deg", "max_slope_deg"]] = [[s["mean"], s["max"]] for s in slope_stats]

# ANNOTATE: which municipality does this line remove, and why is removing it correct rather than a cover-up?
valid_munis = munis_p[munis_p["pixel_count"] > 0]
valid_munis[["name", "mean_elev_m", "mean_slope_deg", "max_slope_deg"]].sort_values(
    "mean_slope_deg", ascending=False
).head(5)


### Extend

Compute the correlation between `mean_elev_m` and `mean_slope_deg` across the 22 valid municipalities with `valid_munis[["mean_elev_m", "mean_slope_deg"]].corr()`. Is it strongly positive, strongly negative, or near zero, and is that consistent with the independence described above?


-----

## Readiness check for Lab 5

Lab 5 applies the above four operations to a DEM of three Shenandoah Valley counties. Confirm as a team that you can do each of the following without looking at this notebook:

| Problem | Lab 5 task | Can your team... |
|---|---|---|
| 1, global | Task C | Compute a raster's minimum, maximum, mean, and a percentile with the `np.nan*` functions, and use one of them in a cell-by-cell comparison? |
| 2, focal | Task C | Build a 3-by-3 window array from a nan-padded raster, reduce it to a focal mean and a focal range, and restore nodata afterward? |
| 3, local | Task D | Convert units, combine two boolean rasters with `&`, convert a cell count to km² using the file's own cell size, and reclassify with `np.digitize` without letting nodata land in the top class? |
| 4, slope | Task E | Compute slope in degrees from a raster whose nodata is already `nan`, and explain why the order matters? |
| 5, zonal | Task F | Reproject a polygon layer to the raster's CRS, run `zonal_stats` on two different rasters, and recognize a zone with zero pixels? |




## Wrap-up and next steps

**Practice** (no submission):

1. Re-run the elevation zonal statistics with `stats=["median", "std"]` added and compare each municipality's mean to its median. Are they close, or does any municipality show a skewed distribution?
2. Run zonal statistics on `focal_range` instead of slope. Does the municipality with the highest mean focal range match the one with the highest mean slope?
3. Using `munis_p`, find the municipality with the largest `pixel_count` and the one with the smallest nonzero `pixel_count`. What does each say about how much of that municipality's area falls inside Portage County?


**Next week:** mapping and visualization: choropleths, then interactive web maps.